# Spam Classification
Works on **any text dataset with a label column** (e.g. the SMS Spam Collection `spam.csv`, email datasets, review datasets). A file explorer opens, then you choose the **text column** and the **label column**.

**Pipeline:** load data -> clean text -> convert text to numbers (Bag-of-Words / TF-IDF) -> train classifiers (Naive Bayes, Logistic Regression, SVM, Random Forest) -> evaluate (accuracy, precision, recall, F1, ROC) -> predict new messages.

**Naive Bayes idea:** $P(\text{spam}\mid \text{words}) \propto P(\text{spam}) \prod_i P(w_i \mid \text{spam})$

## 1. Import Libraries

In [ ]:
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc,
                             precision_recall_curve, ConfusionMatrixDisplay)

np.random.seed(42)

## 2. Select Dataset from File Explorer

In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
file_path = filedialog.askopenfilename(
    title="Select your spam dataset",
    filetypes=[("Data files", "*.csv *.xlsx *.xls *.txt *.tsv"), ("All files", "*.*")]
)
root.destroy()

if not file_path:
    raise SystemExit("No file selected. Re-run this cell.")

print("Selected file:", file_path)

def read_any(path):
    if path.endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    # many spam datasets are not UTF-8 (e.g. spam.csv is latin-1), so try several encodings
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(path, sep=None, engine='python', encoding=enc)
        except Exception:
            continue
    raise ValueError("Could not read the file")

df = read_any(file_path)
print("Shape:", df.shape)
df.head()

## 3. Choose Text and Label Columns

In [ ]:
print("Columns:")
for i, c in enumerate(df.columns):
    sample = str(df[c].iloc[0])[:50]
    print(f"{i}: {c}  | unique = {df[c].nunique()} | e.g. {sample!r}")

def pick(prompt):
    v = input(prompt).strip()
    return df.columns[int(v)] if v.isdigit() and v not in df.columns else v

text_col  = pick("\nTEXT column (the message / email body) - name or index: ")
label_col = pick("LABEL column (spam / ham) - name or index: ")

data = df[[text_col, label_col]].dropna().copy()
data.columns = ['text', 'label_raw']
data['text'] = data['text'].astype(str)

print("\nLabel values found:", data['label_raw'].unique())
spam_value = input("Which label value means SPAM? (type it exactly): ").strip()

matches = [u for u in data['label_raw'].unique() if str(u) == spam_value]
if not matches:
    raise ValueError("Value not found. Re-run this cell and type one of: " + str(list(data['label_raw'].unique())))

data['label'] = (data['label_raw'] == matches[0]).astype(int)   # 1 = spam, 0 = not spam (ham)
data = data.drop_duplicates(subset='text').reset_index(drop=True)
print("\nAfter removing duplicate messages:", data.shape)

## 4. Explore the Data

In [ ]:
counts = data['label'].value_counts().rename({0: 'Ham (not spam)', 1: 'Spam'})
print(counts)
print(f"\nSpam percentage: {data['label'].mean()*100:.2f}%  (class imbalance matters for evaluation)")

data['length'] = data['text'].str.len()
data['n_words'] = data['text'].str.split().str.len()
data['n_digits'] = data['text'].str.count(r'\d')
data['n_upper'] = data['text'].str.count(r'[A-Z]')
data['has_url'] = data['text'].str.contains(r'http|www\.', case=False, regex=True).astype(int)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
counts.plot(kind='bar', ax=ax[0], color=['lightgreen', 'salmon'], rot=0); ax[0].set_title('Class distribution')
sns.histplot(data=data, x='length', hue='label', bins=40, ax=ax[1], palette=['green', 'red'])
ax[1].set_title('Message length (0 = ham, 1 = spam)')
data.groupby('label')[['n_words', 'n_digits', 'n_upper', 'has_url']].mean().T.plot(kind='bar', ax=ax[2], rot=0)
ax[2].set_title('Average features by class')
plt.tight_layout(); plt.show()

## 5. Text Cleaning

In [ ]:
def clean_text(t):
    t = t.lower()
    t = re.sub(r'http\S+|www\.\S+', ' urltoken ', t)        # replace links with a token
    t = re.sub(r'\d+', ' numtoken ', t)                      # replace numbers with a token
    t = t.translate(str.maketrans('', '', string.punctuation))
    t = re.sub(r'\s+', ' ', t).strip()
    return t

data['clean'] = data['text'].apply(clean_text)
data[['text', 'clean', 'label']].head()

## 6. Most Frequent Words: Spam vs Ham

In [ ]:
def top_words(texts, k=15):
    cv = CountVectorizer(stop_words='english')
    X = cv.fit_transform(texts)
    freq = np.asarray(X.sum(axis=0)).ravel()
    idx = freq.argsort()[::-1][:k]
    return pd.Series(freq[idx], index=np.array(cv.get_feature_names_out())[idx])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
top_words(data.loc[data.label == 0, 'clean']).iloc[::-1].plot(kind='barh', ax=ax[0], color='green'); ax[0].set_title('Top words in HAM')
top_words(data.loc[data.label == 1, 'clean']).iloc[::-1].plot(kind='barh', ax=ax[1], color='red');   ax[1].set_title('Top words in SPAM')
plt.tight_layout(); plt.show()

## 7. Train-Test Split (stratified keeps the spam ratio the same)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data['clean'], data['label'], test_size=0.2, random_state=42, stratify=data['label'])

print("Train:", len(X_train), "| Test:", len(X_test))
print("Spam %% in train: %.2f | in test: %.2f" % (y_train.mean()*100, y_test.mean()*100))

## 8. Text to Numbers: Bag-of-Words vs TF-IDF

In [ ]:
bow = CountVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))
tfidf = TfidfVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2), sublinear_tf=True)

Xtr_bow = bow.fit_transform(X_train);   Xte_bow = bow.transform(X_test)
Xtr_tf  = tfidf.fit_transform(X_train); Xte_tf  = tfidf.transform(X_test)

print("Vocabulary size (BoW):", len(bow.vocabulary_))
print("Matrix shape (train):", Xtr_tf.shape, "| sparsity: %.2f%%" % (100 * (1 - Xtr_tf.nnz / np.prod(Xtr_tf.shape))))

## 9. Train Several Classifiers

In [ ]:
def evaluate(name, model, Xtr, Xte):
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
    }, pred

models = {
    'Multinomial NB (BoW)':     (MultinomialNB(alpha=0.5),                          Xtr_bow, Xte_bow),
    'Multinomial NB (TF-IDF)':  (MultinomialNB(alpha=0.1),                          Xtr_tf,  Xte_tf),
    'Bernoulli NB (BoW)':       (BernoulliNB(alpha=0.5),                            Xtr_bow, Xte_bow),
    'Logistic Regression':      (LogisticRegression(max_iter=2000, C=10, class_weight='balanced'), Xtr_tf, Xte_tf),
    'Linear SVM':               (LinearSVC(C=1, class_weight='balanced'),           Xtr_tf,  Xte_tf),
    'Random Forest':            (RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, class_weight='balanced'), Xtr_tf, Xte_tf),
}

results, preds = [], {}
for name, (m, Xtr, Xte) in models.items():
    r, p = evaluate(name, m, Xtr, Xte)
    results.append(r); preds[name] = p

res_df = pd.DataFrame(results).set_index('Model').sort_values('F1', ascending=False)
display(res_df.round(4))

## 10. Model Comparison

In [ ]:
res_df.plot(kind='bar', figsize=(12, 5), rot=25)
plt.ylim(max(0, res_df.values.min() - 0.05), 1.01)
plt.title('Model comparison'); plt.legend(loc='lower right'); plt.grid(alpha=.3); plt.show()

best_name = res_df['F1'].idxmax()
print("Best model by F1:", best_name)

## 11. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, p) in zip(axes.ravel(), preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, p), display_labels=['Ham', 'Spam']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

print("False Positive = a good (ham) message wrongly marked spam  -> usually the WORST error for email")
print("False Negative = a spam message that slips through")

## 12. Detailed Report for the Best Model

In [ ]:
best_model = models[best_name][0]
print(classification_report(y_test, preds[best_name], target_names=['Ham', 'Spam'], digits=4))

## 13. ROC and Precision-Recall Curves

In [ ]:
def get_scores(m, Xte):
    if hasattr(m, 'predict_proba'):
        return m.predict_proba(Xte)[:, 1]
    return m.decision_function(Xte)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name, (m, Xtr, Xte) in models.items():
    s = get_scores(m, Xte)
    fpr, tpr, _ = roc_curve(y_test, s)
    pr, rc, _ = precision_recall_curve(y_test, s)
    ax[0].plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
    ax[1].plot(rc, pr, label=name)
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set_xlabel('False Positive Rate'); ax[0].set_ylabel('True Positive Rate')
ax[0].set_title('ROC curves'); ax[0].legend(fontsize=7)
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision'); ax[1].set_title('Precision-Recall curves'); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 14. Cross-Validation (more reliable than a single split)

In [ ]:
pipes = {
    'Multinomial NB': Pipeline([('tf', TfidfVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))), ('clf', MultinomialNB(alpha=0.1))]),
    'Logistic Regression': Pipeline([('tf', TfidfVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))), ('clf', LogisticRegression(max_iter=2000, C=10, class_weight='balanced'))]),
    'Linear SVM': Pipeline([('tf', TfidfVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))), ('clf', LinearSVC(class_weight='balanced'))]),
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, pipe in pipes.items():
    sc = cross_val_score(pipe, data['clean'], data['label'], cv=skf, scoring='f1')
    cv_rows.append({'Model': name, 'Mean F1': sc.mean(), 'Std': sc.std()})
pd.DataFrame(cv_rows).set_index('Model').round(4)

## 15. Most Spam-Indicative and Ham-Indicative Words

In [ ]:
nb = models['Multinomial NB (BoW)'][0]
feat = np.array(bow.get_feature_names_out())
log_ratio = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]      # log P(w|spam) - log P(w|ham)
order = log_ratio.argsort()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
pd.Series(log_ratio[order[-15:]], index=feat[order[-15:]]).plot(kind='barh', ax=ax[0], color='red')
ax[0].set_title('Words most associated with SPAM (log odds ratio)')
pd.Series(log_ratio[order[:15]], index=feat[order[:15]]).plot(kind='barh', ax=ax[1], color='green')
ax[1].set_title('Words most associated with HAM')
plt.tight_layout(); plt.show()

## 16. Threshold Tuning (trade-off between false positives and false negatives)

In [ ]:
lr = models['Logistic Regression'][0]
prob = lr.predict_proba(Xte_tf)[:, 1]
rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    p = (prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    rows.append({'Threshold': t, 'Precision': precision_score(y_test, p, zero_division=0),
                 'Recall': recall_score(y_test, p), 'F1': f1_score(y_test, p),
                 'False Positives': fp, 'False Negatives': fn})
pd.DataFrame(rows).set_index('Threshold').round(4)

## 17. Misclassified Messages (error analysis)

In [ ]:
test_df = pd.DataFrame({'text': data.loc[X_test.index, 'text'], 'actual': y_test, 'predicted': preds[best_name]})
errors = test_df[test_df.actual != test_df.predicted]
print(f"{len(errors)} misclassified out of {len(test_df)}\n")
pd.set_option('display.max_colwidth', 120)
errors.head(10)

## 18. Predict New Messages

In [ ]:
final_model = Pipeline([('tf', TfidfVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2), sublinear_tf=True)),
                        ('clf', LogisticRegression(max_iter=2000, C=10, class_weight='balanced'))])
final_model.fit(data['clean'], data['label'])      # train on all data for deployment

def predict_message(msg):
    c = clean_text(msg)
    prob = final_model.predict_proba([c])[0, 1]
    return ('SPAM' if prob >= 0.5 else 'NOT SPAM'), prob

samples = [
    "Congratulations! You have won a FREE prize. Call now to claim your reward!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: your account is suspended. Click http://bit.ly/xyz to verify",
    "Can you send me the notes from yesterday's class?",
]
for s in samples:
    label, p = predict_message(s)
    print(f"[{label:8s}  P(spam)={p:.3f}]  {s}")

# Try your own message
while True:
    msg = input("\nType a message to test (or press Enter to stop): ")
    if not msg.strip():
        break
    label, p = predict_message(msg)
    print(f"-> {label}  (P(spam) = {p:.3f})")

## Key Points (for viva)
- **Text classification** needs text converted to numeric features: **Bag-of-Words** (word counts) or **TF-IDF** (counts down-weighted for words common to all documents: $\text{tf-idf}=\text{tf}\times\log\frac{N}{df}$).
- **Naive Bayes** applies Bayes' theorem with the "naive" assumption that words are conditionally independent given the class. It is fast, works well on small data, and is the classic spam filter. **Laplace/additive smoothing** (`alpha`) avoids zero probabilities for unseen words.
- **Class imbalance:** only a small share of messages are spam, so **accuracy is misleading** (predicting "ham" always gives high accuracy). Use precision, recall, F1, ROC-AUC and the PR curve.
  - $\text{Precision}=\frac{TP}{TP+FP}$, $\text{Recall}=\frac{TP}{TP+FN}$, $F_1=\frac{2PR}{P+R}$
- **False positive** (good mail marked spam) is usually costlier than a false negative, so raise the decision threshold to favour precision.
- **Stratified split** keeps the spam ratio equal in train and test. **Fit the vectorizer on training data only** to avoid data leakage.
- **Preprocessing steps:** lowercase, remove punctuation, replace URLs and numbers with tokens, remove stop words, optional stemming or lemmatization, n-grams.
- **Logistic Regression and Linear SVM** are strong on high-dimensional sparse text data; Random Forest is usually weaker and slower here.